In [ ]:
!pip install -q transformers
from transformers import pipeline
clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
labels = ["water leak","energy waste","waste contamination","procurement compliance","accessibility","wellbeing or governance"]
msgs = ["There is a water leak in Building C that has been running all morning.",
        "I want to report that one of our suppliers may not meet our sustainability policy.",
        "The accessible entrance near the main building has been blocked for two days."]
for m in msgs:
    r = clf(m, labels)
    print(m, "->", r["labels"][0], round(r["scores"][0], 2))

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

There is a water leak in Building C that has been running all morning. -> water leak 0.96
I want to report that one of our suppliers may not meet our sustainability policy. -> wellbeing or governance 0.83
The accessible entrance near the main building has been blocked for two days. -> accessibility 0.95


In [ ]:
def rule_based(m):
    t = m.lower()
    if any(k in t for k in ["leak","pipe","flood"]): return "water leak"
    if any(k in t for k in ["air conditioning","aircon","lights on","heating"]): return "energy waste"
    if any(k in t for k in ["recycl","contaminat"]): return "waste contamination"
    if any(k in t for k in ["supplier","procurement","vendor"]): return "procurement compliance"
    if any(k in t for k in ["accessible","wheelchair","ramp","entrance"]): return "accessibility"
    return "other"

msgs2 = msgs + ["pipe broke near bldg C water everywhere pls fix"]
for m in msgs2:
    r = clf(m, labels)
    print(m, "| zero-shot:", r["labels"][0], round(r["scores"][0], 2), "| rule:", rule_based(m))

There is a water leak in Building C that has been running all morning. | zero-shot: water leak 0.96 | rule: water leak
I want to report that one of our suppliers may not meet our sustainability policy. | zero-shot: wellbeing or governance 0.83 | rule: procurement compliance
The accessible entrance near the main building has been blocked for two days. | zero-shot: accessibility 0.95 | rule: accessibility
pipe broke near bldg C water everywhere pls fix | zero-shot: water leak 0.82 | rule: water leak


In [ ]:
!pip install -q transformers accelerate
from transformers import pipeline
gen = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", device_map="auto")

SYSTEM = """PASTE THE SYSTEM INSTRUCTIONS HERE"""
msgs = ["There is a water leak in Building C that has been running all morning.",
        "I want to report that one of our suppliers may not meet our sustainability policy.",
        "The accessible entrance near the main building has been blocked for two days.",
        "pipe broke near bldg C water everywhere pls fix"]

for m in msgs:
    chat = [{"role":"system","content":SYSTEM},
            {"role":"user","content":f'Message: """{m}"""'}]
    out = gen(chat, max_new_tokens=300, do_sample=False)[0]["generated_text"][-1]["content"]
    print(m, "\n", out, "\n")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

There is a water leak in Building C that has been running all morning. 
 Subject: Water Leak Report

Dear [Recipient's Name],

I am writing to report an ongoing issue with the water supply in Building C. The leak, which started this morning, continues unabated and has caused significant damage to various areas of the building.

Please find attached a detailed log of the water usage for the past few days, showing the consistent high consumption despite no apparent use by any known tenants or staff members.

Additionally, I have taken steps to secure the area around the leak to prevent further damage and ensure safety for everyone in the building.

Thank you for your prompt attention to this matter.

Best regards,

[Your Full Name]  
[Your Position]  
[Contact Information] 



[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I want to report that one of our suppliers may not meet our sustainability policy. 
 To report a supplier who may not be meeting your company's sustainability policy, you can follow these steps:

1. **Gather Information**: Collect all relevant details about the supplier and their performance in relation to your sustainability goals. This includes any documented policies or standards they adhere to.

2. **Review Policy Compliance**: Ensure that the supplier is aware of and adheres to your sustainability policy. If necessary, communicate directly with them to confirm compliance.

3. **Document the Issue**: Keep detailed records of the issue, including dates, times, and specific actions taken by both parties.

4. **Discuss with Supplier**: Have an open conversation with the supplier to understand why they might not be meeting expectations. It could be due to lack of resources, outdated practices, or other factors.

5. **Propose Solutions**: Suggest possible solutions such as training prog

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The accessible entrance near the main building has been blocked for two days. 
 Subject: Access to Main Building Entrance Blocked

Dear [Recipient's Name],

I hope this message finds you well.

We regret to inform you that the accessible entrance near the main building has been blocked for the past two days. This is an urgent matter and we are working diligently to resolve it as soon as possible.

Please note that during this time, alternative entrances may be available or temporary access points will be in place. We appreciate your understanding and cooperation while we address this issue promptly.

If you have any questions or require further information, please do not hesitate to contact us at [Contact Information].

Thank you for your attention to this matter.

Best regards,

[Your Full Name]  
[Your Position]  
[Company Name] 

pipe broke near bldg C water everywhere pls fix 
 **System Instructions:** Please report the broken pipe in building C and request immediate repair. Water 